# Linear Regression Health Costs Calculator 🩺

A supervised TensorFlow/Keras regression model for the freeCodeCamp healthcare insurance dataset. The goal is **test MAE below USD 3,500**. Data and results are downloaded/computed when this notebook is run.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split

tf.keras.utils.set_random_seed(42)
if not Path("insurance.csv").exists():
    urlretrieve("https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv",
                "insurance.csv")
dataset = pd.read_csv("insurance.csv")
print(dataset.head())
print(dataset.columns.tolist())


## Encode categorical variables and perform an 80/20 train-test split
The cost target is removed from features before preprocessing. Only training features are used to fit numerical normalization.

In [ ]:
target = "expenses" if "expenses" in dataset.columns else "charges"
labels = dataset[target].astype("float32")
features = pd.get_dummies(dataset.drop(columns=[target]),
                          columns=["sex","smoker","region"], dtype="float32")
features["smoker_age"] = features["age"] * features.get("smoker_yes",0)
features["smoker_bmi"] = features["bmi"] * features.get("smoker_yes",0)
train_dataset, test_dataset, train_labels, test_labels = train_test_split(
    features.astype("float32"), labels, test_size=0.2, random_state=42)
print("Training rows:",len(train_dataset),"Test rows:",len(test_dataset))


## Train a dense regression network
Evaluate MAE directly in dollars using model.evaluate(). The best validation weights are restored to limit overfitting.

In [ ]:
normalizer = tf.keras.layers.Normalization(axis=-1)
normalizer.adapt(train_dataset.to_numpy(dtype="float32"))
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(train_dataset.shape[1],)),
    normalizer,
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1)
])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=.01),
              loss="mae", metrics=["mae"])
history = model.fit(
    train_dataset.to_numpy(dtype="float32"), train_labels.to_numpy(dtype="float32"),
    validation_split=.15, batch_size=32, epochs=450, verbose=0,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_mae", patience=40, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_mae", patience=20, factor=.5, min_lr=1e-5)
    ])
print("Trained epochs:", len(history.history["loss"]))


## Evaluate on held-out test rows
The test set is never used to train or choose weights. No test MAE is claimed in advance.

In [ ]:
loss, mae = model.evaluate(
    test_dataset.to_numpy(dtype="float32"),
    test_labels.to_numpy(dtype="float32"), verbose=0)
predictions = model.predict(test_dataset.to_numpy(dtype="float32"), verbose=0).ravel()
print(f"Unseen test MAE (USD): {mae:,.2f}")
print("PASS" if mae < 3500 else "NOT YET PASSING — tune and retrain")
fig, ax = plt.subplots(figsize=(7,6))
ax.scatter(test_labels, predictions, alpha=.5)
low = min(float(test_labels.min()),float(predictions.min()))
high = max(float(test_labels.max()),float(predictions.max()))
ax.plot([low,high],[low,high],"k--")
ax.set(xlabel="Actual expenses (USD)",ylabel="Predicted expenses (USD)",
       title="Held-out insurance cost predictions")
plt.show()
